### **<font color="green">Context Filter Plugin</font>**

In [1]:
import os
import asyncio

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.plugins.context_filter_plugin import ContextFilterPlugin

import google.genai.types as types

from config import config


# ---------------------------------
# Load Config
# ---------------------------------
PROJECT_ID = config.BQ_PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


# ---------------------------------
# Initialize Gemini
# ---------------------------------
llm = Gemini(
    model="gemini-2.5-flash",
    api_key=config.GOOGLE_API_KEY
)


# ---------------------------------
# Create Context Filter Plugin
# ---------------------------------
context_filter_plugin = ContextFilterPlugin(
    num_invocations_to_keep=3  # Keep only last 3 user invocations
)


# ---------------------------------
# Create Agent
# ---------------------------------
agent = LlmAgent(
    model=llm,
    name="FilteredAgent",
    instruction="""
You are a helpful assistant.
Keep responses concise.
"""
)


# ---------------------------------
# Setup Runner
# ---------------------------------
APP_NAME = "context_filter_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"

session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[context_filter_plugin]  # <-- Plugin attached here
)


# ---------------------------------
# Create Session
# ---------------------------------
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )


# ---------------------------------
# Chat Function
# ---------------------------------
async def chat(user_input: str):
    print(f"\nHuman: {user_input}")

    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )

    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )

    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print(f"AI: {event.content.parts[0].text}")


# ---------------------------------
# Main Demo
# ---------------------------------
async def main():
    await create_session()

    await chat("What is AI?")
    await chat("Explain Machine Learning.")
    await chat("Explain Deep Learning.")
    await chat("Explain Transformers.")
    await chat("Now summarize everything briefly.")


# asyncio.run(main())
await main()


Human: What is AI?
AI: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think like humans and mimic their actions. It encompasses capabilities such as learning, problem-solving, and understanding.

Human: Explain Machine Learning.
AI: Machine Learning (ML) is a subset of AI that enables systems to learn from data, identify patterns, and make decisions or predictions with minimal human intervention. Instead of being explicitly programmed for every task, ML algorithms improve their performance over time through exposure to more data.

Human: Explain Deep Learning.
AI: Deep Learning (DL) is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers (deep neural networks) to learn complex patterns from vast amounts of data. It excels at tasks like image and speech recognition by automatically extracting features from raw input.

Human: Explain Transformers.
AI: Transformers are a type of 

### **<font color="green">Global Instruction Plugin</font>**

In [3]:
import os
import asyncio

from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.plugins.global_instruction_plugin import GlobalInstructionPlugin

import google.genai.types as types
from config import config


# ---------------------------------------
# Load Config
# ---------------------------------------
PROJECT_ID = config.BQ_PROJECT_ID
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"


# ---------------------------------------
# Initialize Gemini
# ---------------------------------------
llm = Gemini(
    model="gemini-2.5-flash",
    api_key=config.GOOGLE_API_KEY
)


# ---------------------------------------
# Define Global Instruction
# ---------------------------------------
global_instruction_plugin = GlobalInstructionPlugin(
    global_instruction="""
You are an elite AI research assistant.
Always respond in a professional tone.
Keep answers concise.
When possible, provide structured bullet points.
"""
)


# ---------------------------------------
# Create Agent (No global instruction here!)
# ---------------------------------------
agent = LlmAgent(
    model=llm,
    name="ProfessionalAgent",
    instruction="You help users understand AI concepts clearly."
)


# ---------------------------------------
# Setup Runner
# ---------------------------------------
APP_NAME = "global_instruction_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"

session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

runner = Runner(
    agent=agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[global_instruction_plugin]  # <-- Attached globally
)


# ---------------------------------------
# Create Session
# ---------------------------------------
async def create_session():
    await session_service.create_session(
        app_name=APP_NAME,
        user_id=USER_ID,
        session_id=SESSION_ID
    )


# ---------------------------------------
# Chat Function
# ---------------------------------------
async def chat(user_input: str):
    print(f"\nHuman: {user_input}")

    content = types.Content(
        role="user",
        parts=[types.Part(text=user_input)]
    )

    events = runner.run(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )

    for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print(f"AI:\n{event.content.parts[0].text}")


# ---------------------------------------
# Main Demo
# ---------------------------------------
async def main():
    await create_session()

    await chat("What is Artificial Intelligence?")
    await chat("Explain Machine Learning.")
    await chat("What are Transformers?")


# asyncio.run(main())
await main()


Human: What is Artificial Intelligence?
AI:
Artificial Intelligence (AI) is a field of computer science dedicated to creating machines capable of performing tasks that typically require human intelligence.

Key aspects include:
*   **Learning**: Acquiring information and rules for using the information.
*   **Reasoning**: Using rules to reach approximate or definite conclusions.
*   **Problem-Solving**: Finding solutions to complex challenges.
*   **Perception**: Interpreting sensory input (e.g., visual, auditory).
*   **Language Understanding**: Processing and generating human language.

Human: Explain Machine Learning.
AI:
Machine Learning (ML) is a subset of Artificial Intelligence that enables systems to learn from data, identify patterns, and make decisions or predictions with minimal human intervention.

Its core principle involves:
*   **Data-Driven Learning**: Algorithms are trained on large datasets, allowing them to automatically discover insights and relationships.
*   **Pa